# A10: Additional data sources (Tanner Edwards)

Personal EDA notebook. Brings three extra sources onto Matt's contract population (`Sorensen-A10.ipynb` §2–3):

| Source | Why |
|---|---|
| The Numbers (`data/raw/the_numbers/`) | Independent budget and **worldwide** gross; pre-release fields (companies, MPAA, source, franchise) |
| Box Office Mojo (`data/raw/boxofficemojo/`) | Daily **domestic** gross; was the film in US theaters at all (S4) |
| RT, stefanoleone992 (`data/raw/stefanoleone992/`) | `actors`, `production_company` |

This notebook only **merges** the sources. No analysis or trimming yet. Solid results move to `A10-master.ipynb`.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")

## 2. Contract population

Same filters as Matt's notebook (cell building `pop`): released, not adult, 2000–2023, budget ≥ $1M, revenue > 0.

In [ ]:
t = pd.read_csv(RAW / "TMDB_movie_dataset_v11.csv", low_memory=False)
t["release_dt"] = pd.to_datetime(t["release_date"], errors="coerce")
t["year"] = t["release_dt"].dt.year

pop = t[
    t["status"].eq("Released") & ~t["adult"] & t["year"].between(2000, 2023)
    & t["budget"].ge(1_000_000) & t["revenue"].gt(0)
].copy()
assert len(pop) == 5766 and pop["id"].is_unique, "population doesn't match Matt's notebook"
print(f"Population: {len(pop):,} films")

## 3. Load the new sources

In [ ]:
tn = pd.read_csv(RAW / "the_numbers/Top Movies (Cleaned Data).csv", low_memory=False)
tn["release_dt"] = pd.to_datetime(tn["Release Date"], errors="coerce")
tn["year"] = tn["release_dt"].dt.year

bom_daily = pd.read_csv(RAW / "boxofficemojo/revenues_per_day.csv.gz", parse_dates=["date"])
# `id` in this file is per ROW (film-day), not per film: collapse to one row per title
bom = (bom_daily.groupby("title")
       .agg(bom_first_date=("date", "min"), bom_domestic_gross=("revenue", "sum"),
            bom_days=("date", "nunique"), bom_max_theaters=("theaters", "max"),
            bom_distributor=("distributor", "first"))
       .reset_index())
bom["year"] = bom["bom_first_date"].dt.year

sl = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_movies.csv", low_memory=False)
sl["year"] = pd.to_datetime(sl["original_release_date"], errors="coerce").dt.year

pd.DataFrame({
    "rows": [len(tn), len(bom_daily), len(bom), len(sl)],
}, index=["The Numbers", "BOM film-days", "BOM films", "RT stefanoleone992"])

## 4. Join each source to the population

Rule: normalized title (lowercase, letters and digits only) **and** release year within ±1.
If a film matches more than one source row, keep the closest year. Source rows that match several population films are counted as ambiguous (checked in the next step, not dropped yet).

In [ ]:
def norm_title(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.lower().str.replace(r"[^a-z0-9]", "", regex=True)


def match(src: pd.DataFrame, title_col: str, prefix: str) -> pd.DataFrame:
    """Return one row per matched population id with the source's columns prefixed."""
    s = src.drop(columns="year").add_prefix(f"{prefix}_")
    s["_key"] = norm_title(src[title_col])
    s["_year"] = src["year"]
    s["_row"] = np.arange(len(s))
    p = pop[["id", "title", "year"]].assign(_key=norm_title(pop["title"]))
    m = p.merge(s, on="_key")
    m[f"{prefix}_year_diff"] = (m["year"] - m["_year"]).abs()
    m = m[m[f"{prefix}_year_diff"] <= 1].sort_values(["id", f"{prefix}_year_diff"]).drop_duplicates("id")
    ambiguous = m["_row"].duplicated(keep=False).sum()
    print(f"{prefix:4} matched {len(m):,} / {len(pop):,} ({len(m) / len(pop):.0%}); "
          f"{ambiguous} films share a source row with another film")
    return m.drop(columns=["title", "year", "_key", "_year", "_row"])


tn_m = match(tn, "Movie Name", "tn")
bom_m = match(bom, "title", "bom")
sl_m = match(sl, "movie_title", "sl")

## 5. Merged frame

One row per population film; source columns are blank where there was no match.

In [ ]:
merged = (pop
          .merge(tn_m, on="id", how="left")
          .merge(bom_m, on="id", how="left")
          .merge(sl_m, on="id", how="left"))
assert len(merged) == len(pop) and merged["id"].is_unique

merged["in_tn"] = merged["tn_Movie Name"].notna()
merged["in_bom"] = merged["bom_title"].notna()
merged["in_sl"] = merged["sl_movie_title"].notna()

print(merged[["in_tn", "in_bom", "in_sl"]].mean().map("{:.0%}".format).to_string())
merged.to_csv(CLEAN / "edwards_merged_sources.csv", index=False)   # gitignored; rebuilt on every run
print(f"\nSaved {CLEAN / 'edwards_merged_sources.csv'}: {merged.shape}")

## 6. Next

- Check the ambiguous matches and a hand-sample of each join.
- Trim columns to allowed inputs vs. outcome fields (no post-release inputs).
- Use The Numbers worldwide gross on Matt's 396 films where TMDB revenue = RT domestic.